In [72]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [73]:
df = pd.read_csv("fmnist_small.csv")

In [74]:
X = df.iloc[:, 1:]
Y = df.iloc[:, 0]

In [75]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.2, random_state=42)

In [86]:
# Normalize image pixel (values in range of 0 to 1)
X_train /= 255.0
X_test /= 255.0

In [77]:
class CustomDataset(nn.Module):
    def __init__(self, input_feat, output_feat):
        self.input_feat = torch.tensor(input_feat, dtype=torch.float32).reshape(-1, 1, 28,28)
        self.output_feat = torch.tensor(output_feat, dtype=torch.long)

    def __len__(self):
        return len(self.input_feat)
    
    def __getitem__(self, index):
        return self.input_feat[index], self.output_feat[index]

In [78]:
train_dataset = CustomDataset(X_train.values, Y_train.values)
test_dataset = CustomDataset(X_test.values, Y_test.values)

In [79]:
train_loader = DataLoader(train_dataset, batch_size=32, drop_last=True, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, drop_last=True, shuffle=False)

### NN CLass

In [87]:
class myCNN(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        # Feature Extractor -> it recognizes the patters from the images. 
        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Feature Classifier -> makes the final decision
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64, 10)
        )

    def forward(self, X):
        X = self.features(X)
        X = self.classifier(X)
        return(X)



In [88]:
learning_rate = 0.01
epochs = 100

In [82]:
model = myCNN(1)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
def evaluate_model(model, test_loader, criterion):
    """
    Evaluate the model on test data
    """
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():  # Disable gradient computation
        for input_feat, output_feat in test_loader:
            outputs = model(input_feat)
            loss = criterion(outputs, output_feat)
            total_loss += loss.item()
            
            # Get predictions
            _, predicted = torch.max(outputs, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(output_feat.cpu().numpy())
    
    avg_loss = total_loss / len(test_loader)
    return avg_loss, all_predictions, all_targets

def calculate_accuracy(predictions, targets):
    predictions = np.array(predictions)
    targets = np.array(targets)
    accuracy = np.mean(predictions == targets) * 100
    return accuracy

def predict_single_image(model, image):
    model.eval()
    with torch.no_grad():
        if len(image.shape) == 3:
            image = image.unsqueeze(0)
        
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)
        probabilities = torch.softmax(outputs, dim=1)
        
    return predicted.item(), probabilities.cpu().numpy()

def predict_batch(model, images):
    model.eval()
    with torch.no_grad():
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        probabilities = torch.softmax(outputs, dim=1)
        
    return predicted.cpu().numpy(), probabilities.cpu().numpy()

In [84]:
train_losses = []
test_losses = []
test_accuracies = []

for epoch in range(epochs):
    model.train()  # Set to training mode
    total_epochs_loss = 0
    
    for input_feat, output_feat in train_loader:
        outputs = model(input_feat)
        loss = criterion(outputs, output_feat)
        
        optimizer.zero_grad()  # Note the parentheses!
        loss.backward()
        optimizer.step()
        
        total_epochs_loss += loss.item()
    
    avg_train_loss = total_epochs_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Evaluate on test set
    test_loss, predictions, targets = evaluate_model(model, test_loader, criterion)
    accuracy = calculate_accuracy(predictions, targets)
    test_losses.append(test_loss)
    test_accuracies.append(accuracy)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.2f}%')

Epoch [10/100], Train Loss: 0.5767, Test Loss: 0.5484, Test Accuracy: 80.46%
Epoch [20/100], Train Loss: 0.3387, Test Loss: 0.4816, Test Accuracy: 82.46%
Epoch [30/100], Train Loss: 0.1267, Test Loss: 0.4933, Test Accuracy: 84.04%
Epoch [40/100], Train Loss: 0.0636, Test Loss: 0.5604, Test Accuracy: 84.29%
Epoch [50/100], Train Loss: 0.0401, Test Loss: 0.6138, Test Accuracy: 84.52%
Epoch [60/100], Train Loss: 0.0261, Test Loss: 0.6450, Test Accuracy: 84.69%
Epoch [70/100], Train Loss: 0.0253, Test Loss: 0.7052, Test Accuracy: 84.19%
Epoch [80/100], Train Loss: 0.0194, Test Loss: 0.7075, Test Accuracy: 84.60%
Epoch [90/100], Train Loss: 0.0156, Test Loss: 0.7961, Test Accuracy: 84.46%
Epoch [100/100], Train Loss: 0.0133, Test Loss: 0.7808, Test Accuracy: 84.56%


In [ ]:
print("FINAL EVALUATION ON TEST SET")
print("="*50)

# Run final evaluation - returns (avg_loss, predictions, targets)
test_loss, predictions, targets = evaluate_model(model, test_loader, criterion)
final_accuracy = calculate_accuracy(predictions, targets)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {final_accuracy:.2f}%")

print("\nClassification Report:")
print(classification_report(targets, predictions, digits=3))

print("\nConfusion Matrix:")
print(confusion_matrix(targets, predictions))


FINAL EVALUATION ON TEST SET

Test Loss: 0.7808
Test Accuracy: 84.56%

Classification Report:
              precision    recall  f1-score   support

           0      0.780     0.817     0.798       491
           1      0.978     0.965     0.972       461
           2      0.771     0.758     0.764       488
           3      0.833     0.840     0.837       476
           4      0.721     0.750     0.735       476
           5      0.939     0.921     0.930       482
           6      0.665     0.602     0.632       532
           7      0.917     0.924     0.921       490
           8      0.941     0.959     0.950       466
           9      0.932     0.963     0.947       438

    accuracy                          0.846      4800
   macro avg      0.848     0.850     0.849      4800
weighted avg      0.844     0.846     0.845      4800


Confusion Matrix:
[[401   0  11  22   3   2  44   0   8   0]
 [  1 445   0  12   0   0   1   0   2   0]
 [  8   0 370   3  70   1  33   0   3   0]